# TB-Trust — 08: Physics-gated deferral, and the retake / refer split

The project already compares uncertainty methods head to head by holding the calibrated
probabilities fixed and swapping only the score the deferral policy ranks on (`eval/run.py`).
The physics certificate slots into that same hook, so the comparison is nearly free:

| signal | what it is | what it answers |
|---|---|---|
| `confidence` | `max(p, 1-p)` | baseline |
| `mc_dropout` | predictive spread | how likely is the classifier wrong? |
| `head` | the learned uncertainty head | ” |
| `ensemble` | member disagreement | ” |
| **`physics`** | **certificate margin, in dB** | **could the photo carry the finding at all?** |

The interesting claim is not that the physics wins on AURC. It is that it is **orthogonal**,
because it answers a different question. A case can be easy for the classifier and
uninformative in the photograph — a confident wrong answer waiting to happen — and only the
physics sees that coming.

Hence the second half of this notebook. "Defer" collapses two situations a clinic must
distinguish: *the photograph is bad* (thirty seconds and another shot, and the physics says
which way to move the phone) versus *the photograph is fine and the case is hard* (retaking
produces an identical image and wastes the visit; refer). Nothing in a learned
confidence score tells those apart. The measured channel does, because the floor is a property of the
capture and the residual uncertainty is a property of the case.

In [ ]:
# --- configuration ---------------------------------------------------------
# Every path comes from the environment first, so this notebook runs unmodified
# on Kaggle, locally, or under scripts/test_notebooks.py in CI.
import os

REPO = os.environ.get("TBTRUST_REPO", "/kaggle/working/tb-trust")
DATA = os.environ.get("TBTRUST_DATA", "/kaggle/input/tuberculosis-tb-chest-xray-dataset")
WORK = os.environ.get("TBTRUST_WORK", "/kaggle/working")
REPO_URL = os.environ.get("TBTRUST_REPO_URL", "https://github.com/AIscend-Research/tb-trust.git")

MANIFEST = f"{WORK}/manifest.csv"
OUT = f"{WORK}/outputs"
os.makedirs(OUT, exist_ok=True)

# Working resolution for the physics. This is the single most consequential knob
# in the whole track: the density floor depends on how many pixels a finding
# spans, so a 2 mm miliary nodule is under two pixels at 320 px and the
# certificate correctly -- but uselessly -- calls every image insufficient.
# A phone photographing a 35 cm film at 3000 px gets about 8 px/mm. 1024 is the
# smallest size at which the severity sweep separates properly; drop it only to
# make a CI run cheap.
PHYSICS_SIZE = int(os.environ.get("TBTRUST_PHYSICS_SIZE", "1024"))
N_IMAGES = int(os.environ.get("TBTRUST_PHYSICS_N", "24"))

print("REPO:", REPO, "\nDATA:", DATA, "\nWORK:", WORK)
print("physics size:", PHYSICS_SIZE, " images:", N_IMAGES)

In [ ]:
# Enter the repo and make it importable. The install is skipped when the package
# already resolves, so re-running is cheap.
import importlib.util
import os
import subprocess
import sys

os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
if importlib.util.find_spec("tbtrust") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    importlib.invalidate_caches()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
print("tbtrust ready from", REPO)

In [ ]:
# Build the manifest if an earlier notebook has not already done it.
import subprocess
import sys
from pathlib import Path

if not Path(MANIFEST).exists():
    r = subprocess.run([sys.executable, "scripts/build_manifest.py", "--raw", DATA, "--out", MANIFEST],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    assert r.returncode == 0, "build_manifest failed"

manifest = pd.read_csv(MANIFEST)
print(len(manifest), "images;", manifest["clinic"].value_counts().to_dict())

## 1. Certificates for every image, at every severity

Each manifest image is treated as a clean film, has fiducials painted on, and is
re-photographed through the forward model at a range of capture severities. That keeps capture
quality a controlled knob. If notebook 05 reported a high certifiable rate on your corpus, pass
`--real` instead and invert the archived photographs directly.

In [ ]:
import subprocess
import sys
from pathlib import Path

CERTS = f"{OUT}/certificates.csv"
n_use = min(N_IMAGES, len(manifest))

if not Path(CERTS).exists():
    r = subprocess.run(
        [sys.executable, "scripts/physics_certificates.py",
         "--manifest", MANIFEST, "--out", CERTS,
         "--severities", "0.0,0.25,0.5,0.75,1.0",
         "--size", str(PHYSICS_SIZE), "--limit", str(n_use)],
        capture_output=True, text=True,
    )
    print(r.stdout[-2500:] or r.stderr[-2500:])
    assert r.returncode == 0, r.stderr[-2000:]

certs = pd.read_csv(CERTS)
print(len(certs), "certificates")
certs[["path", "severity", "certificate", "margin_db", "limiting_factor", "triage_action"]].head()

## 2. A classifier to rank against

The comparison needs probabilities. This fits a deliberately small logistic regression on
downsampled pixels of the *degraded* images, so it degrades with capture quality the way a
real model does, and it needs no training run or GPU.

**Swap in your real model here.** After `tbtrust-eval`, replace `prob` with the
temperature-scaled probabilities from the checkpoint and `learned_conf` with whichever
uncertainty signal you are reporting. Everything below is agnostic to where they came from —
which is the point of holding the point prediction fixed across methods.

In [ ]:
from PIL import Image
from sklearn.linear_model import LogisticRegression

from tbtrust.physics.film import simulate


def features(path, severity, seed):
    img = np.asarray(Image.open(path).convert("L").resize((128, 128), Image.BILINEAR))
    photo, _ = simulate(img, severity=float(severity), rng=np.random.default_rng(seed), size=128)
    x = np.asarray(photo, dtype=np.float64) / 255.0
    small = x.reshape(16, 8, 16, 8).mean(axis=(1, 3)).ravel()
    return np.concatenate([small, [x.mean(), x.std()]])


rows = certs.drop_duplicates(subset=["path", "severity"]).reset_index(drop=True)
X = np.array([features(r.path, r.severity, i) for i, r in enumerate(rows.itertuples())])
y = rows["label"].to_numpy().astype(int)

# Split by *image* and stratify by label. Both halves of that matter. Splitting by
# row would put the same film at five severities on both sides -- near-duplicates,
# and a fantasy accuracy. Not stratifying lets a small corpus put every positive on
# one side, which fails outright rather than quietly.
per_image = rows.drop_duplicates("path")[["path", "label"]]
rng = np.random.default_rng(0)
train_paths = set()
for _lab, grp in per_image.groupby("label"):
    ps = grp["path"].to_numpy(dtype=object)
    rng.shuffle(ps)
    train_paths.update(ps[: max(1, round(0.6 * len(ps)))])
tr = rows["path"].isin(train_paths).to_numpy()
assert rows.loc[tr, "label"].nunique() > 1 and rows.loc[~tr, "label"].nunique() > 1, (
    "both splits need both classes; raise TBTRUST_PHYSICS_N")

clf = LogisticRegression(max_iter=2000, C=0.5).fit(X[tr], y[tr])
rows["prob"] = clf.predict_proba(X)[:, 1]
rows["is_test"] = ~tr
print(f"stand-in classifier: train acc {clf.score(X[tr], y[tr]):.2f}, "
      f"test acc {clf.score(X[~tr], y[~tr]):.2f} on {(~tr).sum()} held-out rows")

## 3. Does the certificate margin respond to capture quality?

The same premise check `eval/degradation_uncertainty.py` applies to the learned signals: the
"retake the photo" message is only justified for a signal that actually falls as the photo gets
worse. The certificate should pass by construction, since it is computed *from* the capture —
so a weak correlation here means something is broken upstream, most likely the fiducial
detector failing on degraded images and silently abstaining.

In [ ]:
from tbtrust.eval import physics_deferral as PD

test = rows[rows["is_test"]].reset_index(drop=True)
resp = PD.severity_response(test["margin_db"], test["severity"])
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in resp.items()})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].scatter(test["severity"] + np.random.default_rng(0).normal(0, 0.012, len(test)),
                test["margin_db"], alpha=0.5, s=18)
axes[0].axhline(0, color="k", ls="--", lw=1)
axes[0].set_xlabel("capture severity")
axes[0].set_ylabel("certificate margin (dB)")
axes[0].set_title(f"Physics responds to capture quality\n(Spearman {resp['spearman']:.2f})")

conf = np.maximum(test["prob"], 1 - test["prob"])
axes[1].scatter(test["margin_db"], conf, alpha=0.5, s=18)
axes[1].set_xlabel("certificate margin (dB)")
axes[1].set_ylabel("model confidence")
axes[1].set_title("Physics vs learned: two different questions")
fig.tight_layout()
plt.show()

## 4. Three deferral policies on one fixed set of probabilities

- **learned** — rank on the model's confidence, defer the tail.
- **physics** — veto every image the certificate calls insufficient.
- **gated** — veto on the certificate first, then rank the survivors by the learned score.

The gated policy is the one to deploy, and it matches the clinical order of operations: first
ask whether the image is usable, then ask whether the finding is ambiguous.

The threshold is tuned on the training split and *applied* to the test split, never re-tuned —
the same discipline `eval/run.py` enforces, and the difference between a real number and an
inflated one.

In [ ]:
from tbtrust.eval import deferral as D

train = rows[~rows["is_test"]]
op = D.tune_threshold(train["label"], train["prob"], target="accuracy",
                      target_value=0.95, min_coverage=0.5)
print(f"threshold tuned on train: T*={op.threshold:.3f} (train coverage {op.coverage:.2f})")

res = PD.compare_policies(
    labels=test["label"], probs=test["prob"], margins_db=test["margin_db"],
    abstained=test["abstained"], threshold=op.threshold,
)
policies = pd.DataFrame([r.as_dict() for r in res]).set_index("name")
display(policies[["aurc", "coverage", "accuracy", "sensitivity", "specificity",
                  "n_deferred", "deferred_wrong_frac", "kept_wrong_frac"]].round(3))

In [ ]:
comp = PD.complementarity(test["label"], test["prob"], test["margin_db"], abstained=test["abstained"])
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in comp.items()})

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

phys_conf = PD.physics_confidence(test["margin_db"], test["abstained"])
D.plot_risk_coverage(test["label"], test["prob"], ax=axes[0], label="learned")
D.plot_risk_coverage(test["label"], test["prob"], ax=axes[0], label="physics", confidence=phys_conf)
insuff = (test["margin_db"] <= 0) | test["abstained"]
gated = np.where(insuff, 0.0, np.maximum(test["prob"], 1 - test["prob"]))
D.plot_risk_coverage(test["label"], test["prob"], ax=axes[0], label="physics-gated", confidence=gated)
axes[0].set_title("Risk-coverage by ranking signal")

labels = ["caught by\nboth", "physics\nonly", "learned\nonly"]
vals = [comp["errors_caught_by_physics"] - comp["errors_only_physics"],
        comp["errors_only_physics"], comp["errors_only_learned"]]
axes[1].bar(labels, vals, color=["grey", "tab:red", "tab:blue"])
axes[1].set_ylabel("errors flagged")
axes[1].set_title(f"Complementarity (Jaccard {comp['jaccard']:.2f})\n"
                  "errors only the physics catches are the\nconfident-and-wrong cases")
fig.tight_layout()
plt.show()

## 5. Triage: retake, refer, or report

The deliverable is not a flag, it is an **instruction**. `glare.hotspot` already knows whether
the veil is a specular blob that moves when the phone moves or a diffuse wash that does not,
and `psf.PSFEstimate.anisotropy` already knows whether the blur is directional shake or
symmetric defocus. Those turn "the image is bad" into "step to your left" or "hold still" or
"shade the lightbox" — the difference between a retake that helps and one that reproduces the
same photograph.

`retake_rate` is the operationally load-bearing number. A policy that flags 60% of images for
retake will be switched off within a week however good its physics is.

In [ ]:
tv = PD.triage_value(test["triage_action"], test["label"], test["prob"])
print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in tv.items()})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mix = test.groupby(["severity", "triage_action"]).size().unstack(fill_value=0)
mix = mix.div(mix.sum(axis=1), axis=0)
mix.plot(kind="bar", stacked=True, ax=axes[0], rot=0)
axes[0].set_ylabel("fraction")
axes[0].set_title("Triage action vs capture severity")
axes[0].legend(fontsize=8)

reasons = test[test["triage_action"] == "retake"]["triage_reason"].value_counts()
if len(reasons):
    axes[1].barh(reasons.index, reasons.to_numpy())
axes[1].set_title("Why a retake was requested")
axes[1].set_xlabel("images")
fig.tight_layout()
plt.show()

The figure below is the argument in one picture: a measured glare *field*, the direction the
reflection sits in, and an arrow saying which way to move. A confidence score can say "this
image is bad"; only a measured channel can say "step to your right", and that difference is
what makes a retake worth the patient's time instead of reproducing the same photograph.

In [ ]:
from tbtrust.physics import figures as FIG
from tbtrust.physics.certificate import certify as _certify
from tbtrust.physics.film import simulate as _simulate
from tbtrust.physics.invert import invert as _invert
from tbtrust.physics.triage import triage as _triage

# Re-run one flagged image so the veil field and the PSF are in hand for the figure.
worst = test.sort_values("margin_db").iloc[0]
_img = np.asarray(Image.open(worst.path).convert("L"))
_photo, _ = _simulate(_img, severity=float(worst.severity),
                      rng=np.random.default_rng(0), size=PHYSICS_SIZE)
_cal = _invert(_photo)
_cert = _certify(_cal)
_dec = _triage(_cert, _cal, model_confidence=float(max(worst.prob, 1 - worst.prob)))

FIG.show(FIG.retake_instruction(_cal, _dec))
plt.show()
print(_dec.action.value.upper(), "-", _dec.reason)

In [ ]:
# The instructions themselves. These are what an operator would see, and they are the
# clearest demonstration that the physics is doing something a confidence score cannot.
seen = set()
for r in test.sort_values("margin_db").itertuples():
    if r.triage_action == "report" or r.triage_reason in seen:
        continue
    seen.add(r.triage_reason)
    print(f"[{r.triage_action.upper()}]  margin {r.margin_db:+.1f} dB, limited by {r.limiting_factor}"
          f"{f', expected gain {r.expected_retake_gain_db:.1f} dB' if r.triage_action == 'retake' else ''}")
    print(f"    {r.triage_instruction}\n")

## 6. What to report

Save the tables. For the paper, the four numbers that matter are:

1. **Fiducial coverage** per clinic (notebook 05) — it bounds every claim here.
2. **Calibration of the bound**, `predicted / empirical` from notebook 07 — a bound quoted
   without its measured calibration is an assertion.
3. **Complementarity**, above — errors the physics catches that no learned signal does. This
   is the argument for the method even where it loses on AURC, because those are the
   confident-and-wrong cases a screening system most needs to catch.
4. **Retake rate and its split**, above — whether a clinic could actually live with the policy.

And the limitation to state proactively, because a reviewer will: this is all in silico. The
degradation is simulated, the finding contrasts in `physics/findings.py` are nominal
placeholders, and the veil is measured around the edge of the field and interpolated across the
middle, so a dim central reflection is under-reported and the certificate is optimistic there.
`docs/PHYSICS.md` and `docs/LIMITATIONS.md` carry the full list.

In [ ]:
policies.to_csv(f"{OUT}/physics_policy_comparison.csv")
pd.Series(comp).to_frame("value").to_csv(f"{OUT}/physics_complementarity.csv")
pd.Series(tv).to_frame("value").to_csv(f"{OUT}/physics_triage_value.csv")
test.to_csv(f"{OUT}/physics_deferral_rows.csv", index=False)
print("wrote:", *[f"{OUT}/physics_{n}.csv" for n in
                  ("policy_comparison", "complementarity", "triage_value", "deferral_rows")], sep="\n  ")